# Pre-Training LC-Phase: Long-Context Extension

**Paper §2.5** — after the main pre-training run the model is continuously trained
on a mixture of very long sequences (up to 512 K tokens) and normal-length sequences
(4 K tokens) to extend its effective context window without degrading short-context
benchmarks.

> Using only very long sequences degraded short-context benchmarks; mixing in
> 4 K sequences preserves them.

A **constant** learning rate of 1e-5 is used throughout.

### Notebook contents
1. Environment setup
2. Imports & hyperparameters
3. Tokenizer
4. Dataset
5. Model (load Phase 2 checkpoint)
6. Optimizer (constant LR)
7. Checkpoint manager
8. Training loop


## 0. Environment Setup

Detects Colab / Kaggle, installs packages, and adds the repo to `sys.path`.

In [ ]:
import os, sys

IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


In [ ]:
if IN_COLAB or IN_KAGGLE:
    # Set ACCELERATOR to "cuda12" (GPU) or "tpu" (Colab TPU only).
    ACCELERATOR = "cuda12"
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         f"jax[{ACCELERATOR}]", "flax", "optax", "orbax-checkpoint",
         "datasets", "transformers", "matplotlib"],
        check=True,
    )


In [ ]:
import pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"


def _detect_repo_root() -> pathlib.Path:
    """Detect local repo root by searching upward for key project files."""
    cwd = pathlib.Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "training_shared.py").exists() and (candidate / "nemotron.py").exists():
            return candidate
    return cwd


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    REPO_DIR = _detect_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("REPO_DIR:", REPO_DIR)
print(f"NUM_DEVICES={NUM_DEVICES}  (batch is sharded {BATCH_SIZE // NUM_DEVICES} samples/device)")


## 1. Imports & Hyperparameters

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import numpy as np
from datasets import load_dataset
from flax import nnx
from transformers import AutoTokenizer

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock
from training_shared import (
    LC_SEQ_LEN, LC_PHASE_STEPS, LC_PHASE_LR,
    PRETRAIN_WD, PRETRAIN_B1, PRETRAIN_B2,
    PRETRAIN_BATCH, PRETRAIN_CKPT_DIR,
    build_model, collect_moe_layers, make_constant_lr_optimizer,
    load_pretrain_data, make_batches, pretrain_step, update_moe_biases,
    make_checkpoint_manager, save_checkpoint, try_load_from_dir,
    NUM_DEVICES,
)

print("JAX devices:", jax.devices())


In [ ]:
SEQ_LEN     = LC_SEQ_LEN     # 512
TRAIN_STEPS = LC_PHASE_STEPS # 100
BATCH_SIZE  = PRETRAIN_BATCH # 2
CKPT_DIR    = PRETRAIN_CKPT_DIR

print(f"LC-Phase: SEQ_LEN={SEQ_LEN} | STEPS={TRAIN_STEPS} | LR={LC_PHASE_LR}")
print(f"NUM_DEVICES={NUM_DEVICES}  ({BATCH_SIZE // NUM_DEVICES} samples/device)")


## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {tokenizer.vocab_size}")


## 3. Dataset

The LC-Phase mixes long (512-token, here) and normal sequences.
In the paper, sequences up to 512 K tokens are used; we sample a fresh non-overlapping slice.

In [ ]:
lc_chunks = load_pretrain_data(
    split="train", max_samples=100, seq_len=SEQ_LEN, tokenizer=tokenizer, skip=700
)

print(f"LC chunks : {len(lc_chunks)}  shape={lc_chunks.shape}")


## 4. Model — Load Phase 2 Checkpoint

In [ ]:
print("Building model ...")
model = build_model(seed=0)
moe_layers = collect_moe_layers(model)

loaded = try_load_from_dir(PRETRAIN_CKPT_DIR, model, model.config)
if loaded:
    print("Resumed from pretrain checkpoint.")
else:
    print("No checkpoint found — starting from scratch.")


## 5. Optimizer (Constant LR)

In [ ]:
tx = make_constant_lr_optimizer(LC_PHASE_LR, PRETRAIN_WD, PRETRAIN_B1, PRETRAIN_B2)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)
print(f"Constant-LR AdamW created (lr={LC_PHASE_LR}).")


## 6. Checkpoint Manager

In [ ]:
ckpt_mgr = make_checkpoint_manager(CKPT_DIR)
print(f"Checkpoints → {CKPT_DIR}")


## 7. Training Loop

In [ ]:
print(f"\n=== Pre-Training LC-Phase: Long-Context Extension ===")
print(f"Training for {TRAIN_STEPS} steps (seq_len={SEQ_LEN}) ...\n")

loss_history: list[float] = []
step = 0
batch_iter = iter(make_batches(lc_chunks, BATCH_SIZE))

while step < TRAIN_STEPS:
    try:
        batch_np = next(batch_iter)
    except StopIteration:
        batch_iter = iter(make_batches(lc_chunks, BATCH_SIZE))
        batch_np = next(batch_iter)

    batch = batch_np  # already sharded across all devices by make_batches
    loss  = pretrain_step(model, optimizer, batch)
    update_moe_biases(moe_layers)
    step += 1
    loss_history.append(float(loss))

    if step % 50 == 0:
        print(f"  LC-Phase step {step:3d} | loss={float(loss):.4f}")

# Offset step key to avoid collision with Phase 1/2 checkpoint keys.
save_checkpoint(ckpt_mgr, model, step + 10_000)
print("\nLC-Phase complete.")


## 8. Loss Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history)
    plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("LC-Phase Training Loss")
    plt.grid(True); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")
